In [1]:
import json

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

from shapely.geometry import Polygon

Load census variables and parse for relevant information

In [3]:
# Create mapping for characteristic IDs to column names
characteristic_mapping = {
    1: 'CHAR_POP21'  # Population, 2021
}

df_cen_cma_data = pd.read_csv('../../data/census/98-401-X2021012_English_CSV_data.csv', encoding='latin')
df_cen_cma_data = df_cen_cma_data[['DGUID', 'GEO_LEVEL', 'GEO_NAME', 'CHARACTERISTIC_ID', 'C1_COUNT_TOTAL']]

# Filter for only the characteristics we want
df_cen_cma_data = df_cen_cma_data[df_cen_cma_data['CHARACTERISTIC_ID'].isin(characteristic_mapping.keys())]

# Map characteristic IDs to column names
df_cen_cma_data['CHARACTERISTIC_COLUMN'] = df_cen_cma_data['CHARACTERISTIC_ID'].map(characteristic_mapping)

# Pivot to create separate columns for each characteristic
df_cen_cma_data = df_cen_cma_data.pivot_table(
    index=['DGUID', 'GEO_LEVEL', 'GEO_NAME'], 
    columns='CHARACTERISTIC_COLUMN', 
    values='C1_COUNT_TOTAL',
    aggfunc='first'
).reset_index()

# Flatten column names
df_cen_cma_data.columns.name = None

In [4]:
df_ada_cma_rel = pd.read_csv('../../data/census/ada_cma_relation.csv')
df_ada_cma_rel = df_ada_cma_rel.rename(columns={'CMADGUID_RMRIDUGD': 'CMADGUID', 'ADADGUID_ADAIDUGD': 'ADADGUID'})

In [5]:
# lcma000b21a_e is a folder containing the shapefile components of census metropolitan areas of Canada 2021
gdf_cma = gpd.read_file('../../data/census/lcma000b21a_e')
gdf_cma = gdf_cma[['CMAUID', 'DGUID', 'CMANAME', 'PRUID', 'geometry']]

# Merge duplicates on DGUID: first row's attrs + unioned geometry
gdf_cma = (
    gdf_cma
    .groupby('DGUID', as_index=False)
    .agg({
        'CMAUID': 'first',
        'CMANAME': 'first',
        'PRUID': 'first',
        'geometry': lambda x: x.unary_union
    })
)

# Remove anything inside parentheses (and the parentheses themselves)
gdf_cma['CMANAME'] = gdf_cma['CMANAME'].str.replace(r"\s*\(.*?\)", "", regex=True).str.strip()

gdf_cma = gpd.GeoDataFrame(gdf_cma, geometry='geometry')
if gdf_cma.crs is None:
    gdf_cma.set_crs("EPSG:3347", inplace=True)

Load tariff information and join to CMAs

In [6]:
# Load ADA-level tariff counts and percents
df_tariffs_ada_count = pd.read_excel('../raw/tariff-impacts-ada-data_8_27_2026.xlsx', sheet_name='Counts').drop(columns=['geometry'])
df_tariffs_ada_pct = pd.read_excel('../raw/tariff-impacts-ada-data_8_27_2026.xlsx', sheet_name='Percents')


In [7]:
df_tariffs_count_filtered = (
    df_tariffs_ada_count
    # .drop(columns=['geometry'])
    .merge(df_ada_cma_rel, on='ADADGUID', how='left')
    .dropna(subset=['CMADGUID'])
)

# Group by CMADGUID and sum all tariff columns
tariff_columns_count = [col for col in df_tariffs_count_filtered.columns if col not in ['ADADGUID', 'CMADGUID']]
df_tariffs_cma_count = df_tariffs_count_filtered.groupby('CMADGUID')[tariff_columns_count].sum().reset_index()

print(f"Shape of CMA tariffs (counts) dataframe: {df_tariffs_cma_count.shape}")
df_tariffs_cma_count.head()

Shape of CMA tariffs (counts) dataframe: (152, 55)


,CMADGUID,Auto_B,Alum_B,Steel_B,Cop_B,Lum_B,Ene_B,MHDV_B,CUSMA_B,Total_B,...,after August 22_C,Section 338_B,Section 338_E,Section 338_C,CUSMA_B.1,CUSMA_E.1,CUSMA_C.1,Total_B.1,Total_E.1,Total_C.1
0,2021S0503001,7,17,14,2,20,10,1,118,118,...,1809,69,637,1203,118,2046,2736,118,2046,2736
1,2021S0503205,35,89,56,10,44,39,0,401,416,...,9436,259,3424,6446,400,6270,10711,417,7104,11698
2,2021S0503305,15,37,34,5,21,19,2,177,191,...,3461,96,2251,2309,172,3584,4477,191,4324,4975
3,2021S0503310,5,17,13,1,18,9,1,136,142,...,4048,90,2131,2627,136,2866,4052,142,3893,5181
4,2021S0503320,9,26,15,0,22,10,1,126,132,...,1703,72,595,1159,126,1666,2414,132,1832,2567


In [8]:
# Melt percents and counts to long format, extract base and numeric suffix
def melt_tariffs(df, value_name, suffix_map=None):
    df_long = df.melt(id_vars=['ADADGUID'], var_name='tariff', value_name=value_name)
    df_long['base'] = df_long['tariff'].str.replace(r'_(1|2|3|B|E|C)$', '', regex=True)
    df_long['suffix'] = df_long['tariff'].str.extract(r'_([1-3BEC])$')[0]
    if suffix_map:
        df_long['suffix'] = df_long['suffix'].map(suffix_map)
    return df_long

suffix_map_counts = {'B': '1', 'E': '2', 'C': '3'}

df_pct_long = melt_tariffs(df_tariffs_ada_pct, 'percent')
df_count_long = melt_tariffs(df_tariffs_ada_count, 'count', suffix_map=suffix_map_counts)

# Merge percents and counts, add CMA IDs
df_merge = (
    df_pct_long
    .merge(df_count_long, on=['ADADGUID', 'base', 'suffix'], how='left')
    .merge(df_ada_cma_rel, on='ADADGUID', how='left')
    .dropna(subset=['CMADGUID'])
)

# Compute weighted percent and aggregate per CMA
df_cma = (
    df_merge.assign(weighted=lambda x: x['percent'] * x['count'])
    .groupby(['CMADGUID', 'base', 'suffix'], observed=True)
    .agg(total_weighted=('weighted', 'sum'), total_count=('count', 'sum'))
    .reset_index()
)
df_cma['cma_percent'] = (df_cma['total_weighted'] / df_cma['total_count']) * 100
df_cma.loc[df_cma['total_count'] == 0, 'cma_percent'] = np.nan

# Pivot to wide format
df_cma['colname'] = df_cma['base'] + '_' + df_cma['suffix']
df_tariffs_cma_pct = df_cma.pivot(index='CMADGUID', columns='colname', values='cma_percent').reset_index()

print(f"Shape of CMA tariffs (percents) dataframe: {df_tariffs_cma_pct.shape}")
df_tariffs_cma_pct.head()

Shape of CMA tariffs (percents) dataframe: (152, 49)


colname,CMADGUID,Alcohol_1,Alcohol_2,Alcohol_3,Alum_1,Alum_2,Alum_3,Auto_1,Auto_2,Auto_3,...,Steel_3,Total_1,Total_2,Total_3,after August 22_1,after August 22_2,after August 22_3,before August 22_1,before August 22_2,before August 22_3
0,2021S0503001,0.434994,0.313909,0.049211,0.569074,0.706492,0.223803,0.510324,0.210099,0.151038,...,0.215348,2.111118,7.512447,2.625433,1.723631,1.642633,1.771159,1.220461,1.345830,0.888897
1,2021S0503205,0.837688,0.422927,0.164547,1.282373,3.215483,1.225206,0.609318,1.269459,0.518583,...,0.702108,6.173923,6.182828,4.749137,4.654020,5.878024,4.081456,2.284452,4.883965,2.704081
2,2021S0503305,0.376059,0.702772,0.207428,1.162877,2.527059,0.899740,0.663729,0.849572,0.531749,...,0.818343,6.062326,9.002227,6.528610,4.301027,8.749673,4.569116,2.588903,4.328603,2.289213
3,2021S0503310,0.968335,5.982111,0.637986,1.065927,2.771979,1.109257,0.901604,3.344821,0.624873,...,2.214564,10.772117,9.166756,8.365134,10.476900,9.530879,6.629609,2.871069,7.125123,4.358254
4,2021S0503320,1.246377,0.683107,0.455458,1.389997,2.001189,0.821503,0.688483,2.629791,0.408295,...,0.700154,6.647012,16.268405,5.584263,4.448338,17.048451,3.811733,2.804928,18.149476,2.248984


In [9]:
# First, we need to aggregate census data from ADA to CMA level
df_cen_ada = df_cen_cma_data.rename(columns={'DGUID': 'ADADGUID'})

# Merge census ADA data with ADA-CMA relationship
df_cen_with_cma = df_cen_ada.merge(df_ada_cma_rel, on='ADADGUID', how='left')

# Aggregate population to CMA level
df_cen_cma = df_cen_with_cma.groupby('CMADGUID').agg({
    'GEO_NAME': 'first',  # We'll update this later with proper CMA names
    'CHAR_POP21': 'sum'
}).reset_index()

# Add GEO_LEVEL
df_cen_cma['GEO_LEVEL'] = 'Census metropolitan area'

# Merge with GDF to get proper CMA names
gdf_cma_names = gdf_cma[['DGUID', 'CMANAME']].rename(columns={'DGUID': 'CMADGUID', 'CMANAME': 'GEO_NAME'})
df_cen_cma = df_cen_cma.drop(columns=['GEO_NAME']).merge(gdf_cma_names, on='CMADGUID', how='left')

# Now merge with tariff data
df_final_counts = df_cen_cma.merge(df_tariffs_cma_count, on='CMADGUID', how='inner')
df_final_percents = df_cen_cma.merge(df_tariffs_cma_pct, on='CMADGUID', how='inner')

print(f"Shape of final counts dataframe: {df_final_counts.shape}")
print(f"Shape of final percents dataframe: {df_final_percents.shape}")


Shape of final counts dataframe: (152, 58)
Shape of final percents dataframe: (152, 52)


In [13]:
# Prepare geometry data
gdf_cma_geom = gpd.GeoDataFrame(
    gdf_cma[['DGUID', 'geometry']].rename(columns={'DGUID': 'CMADGUID'}),
    geometry='geometry',
    crs=gdf_cma.crs
)

# ---- COUNTS ----
gdf_final_counts_full = gpd.GeoDataFrame(
    df_final_counts.merge(gdf_cma_geom, on='CMADGUID', how='inner'),
    geometry='geometry',
    crs=gdf_cma.crs
)

# Centroids for counts
gdf_cma_centroids = gdf_cma_geom.copy()
gdf_cma_centroids['geometry'] = gdf_cma_centroids.geometry.centroid
gdf_cma_centroids = gdf_cma_centroids.to_crs('EPSG:4326')

gdf_final_counts_centroids = gpd.GeoDataFrame(
    df_final_counts.merge(gdf_cma_centroids, on='CMADGUID', how='inner'),
    geometry='geometry',
    crs='EPSG:4326'
)

# Save counts
gdf_final_counts_full.to_file('../../data/cma/cma_tariffs_counts_full_geometryv2.gpkg', driver='GPKG')

# Save counts centroids as CSV: convert geometry to WKT only for the CSV copy
df_counts_centroids_csv = gdf_final_counts_centroids.copy()
df_counts_centroids_csv['geometry'] = df_counts_centroids_csv.geometry.apply(lambda geom: geom.wkt)
df_counts_centroids_csv.to_csv('../../data/cma/cma_tariffs_counts_centroidsv2.csv', index=False)

# ---- PERCENTS ----
gdf_final_percents_full = gpd.GeoDataFrame(
    df_final_percents.merge(gdf_cma_geom, on='CMADGUID', how='inner'),
    geometry='geometry',
    crs=gdf_cma.crs
)

gdf_final_percents_centroids = gpd.GeoDataFrame(
    df_final_percents.merge(gdf_cma_centroids, on='CMADGUID', how='inner'),
    geometry='geometry',
    crs='EPSG:4326'
)

# Save percents
gdf_final_percents_full.to_file('../../data/cma/cma_tariffs_percents_full_geometryv2.gpkg', driver='GPKG')

# Save percents centroids as CSV: convert geometry to WKT only for the CSV copy
df_percents_centroids_csv = gdf_final_percents_centroids.copy()
df_percents_centroids_csv['geometry'] = df_percents_centroids_csv.geometry.apply(lambda geom: geom.wkt)
df_percents_centroids_csv.to_csv('../../data/cma/cma_tariffs_percents_centroidsv2.csv', index=False)

# Print shapes
print(f"Counts full geometry shape: {gdf_final_counts_full.shape}")
print(f"Percents full geometry shape: {gdf_final_percents_full.shape}")
print(f"Counts centroids shape: {gdf_final_counts_centroids.shape}")
print(f"Percents centroids shape: {gdf_final_percents_centroids.shape}")


C:\Users\yihoi\AppData\Local\Temp\ipykernel_8004\1561093363.py:31: UserWarning: Geometry column does not contain geometry.
  df_counts_centroids_csv['geometry'] = df_counts_centroids_csv.geometry.apply(lambda geom: geom.wkt)


Counts full geometry shape: (152, 59)
Percents full geometry shape: (152, 53)
Counts centroids shape: (152, 59)
Percents centroids shape: (152, 53)


C:\Users\yihoi\AppData\Local\Temp\ipykernel_8004\1561093363.py:52: UserWarning: Geometry column does not contain geometry.
  df_percents_centroids_csv['geometry'] = df_percents_centroids_csv.geometry.apply(lambda geom: geom.wkt)


In [14]:
counts_path = '../../data/cma/cma_tariffs_counts_centroidsv2.csv'
percents_path = '../../data/cma/cma_tariffs_percents_centroidsv2.csv'

counts_json_path = '../../data/cma/cma_tariffs_counts_centroidsv2.json'
percents_json_path = '../../data/cma/cma_tariffs_percents_centroidsv2.json'

def csv_to_json_records(csv_path):
    df = pd.read_csv(csv_path)

    # Replace all NaN, NaT, and inf values with None so they become valid JSON null
    df = df.replace({np.nan: None, np.inf: None, -np.inf: None})

    return df.to_dict(orient='records')

counts_records = csv_to_json_records(counts_path)
percents_records = csv_to_json_records(percents_path)

with open(counts_json_path, 'w', encoding='utf-8') as f:
    json.dump(counts_records, f, ensure_ascii=False, indent=4)
with open(percents_json_path, 'w', encoding='utf-8') as f:
    json.dump(percents_records, f, ensure_ascii=False, indent=4)

print(f'Saved counts JSON: {counts_json_path} (rows={len(counts_records)})')
print(f'Saved percents JSON: {percents_json_path} (rows={len(percents_records)})')


Saved counts JSON: ../../data/cma/cma_tariffs_counts_centroidsv2.json (rows=152)
Saved percents JSON: ../../data/cma/cma_tariffs_percents_centroidsv2.json (rows=152)


In [15]:
df_cma_pcts = pd.read_csv(percents_path)
df_cma_pcts = df_cma_pcts[df_cma_pcts['GEO_LEVEL'] == 'Census metropolitan area']
df_cma_pcts.describe(percentiles=[0.2, 0.4, 0.6, 0.8]).round(2).to_csv('../../data/cma/cma_tariffs_percents_variancev2.csv')